In [2]:
import requests
import os

VERCEL_TOKEN = "vcp_4DKKSk3VWHELULMvuTXC0KXgP50m0JTbbols3QLdbpeP0R3ett3ONFTF"
PROJECT_NAME = "nexus-prime"

headers = {
    "Authorization": f"Bearer {VERCEL_TOKEN}",
    "Content-Type": "application/json"
}

# 1. Get Project ID
print(f"Searching for project: {PROJECT_NAME}")
res = requests.get(f"https://api.vercel.com/v9/projects/{PROJECT_NAME}", headers=headers)
if res.status_code != 200:
    print(f"Error fetching project: {res.text}")
    # Try listing projects if exact name match fails
    res = requests.get("https://api.vercel.com/v9/projects", headers=headers)
    projects = res.json().get('projects', [])
    print(f"Available projects: {[p['name'] for p in projects]}")
    # Pick the first one or the one containing nexus
    target = next((p for p in projects if "nexus" in p['name']), None)
    if target:
        PROJECT_ID = target['id']
        PROJECT_NAME = target['name']
    else:
        raise Exception("Project not found on Vercel.")
else:
    PROJECT_ID = res.json()['id']

print(f"Found Project: {PROJECT_NAME} ({PROJECT_ID})")

# 2. Update Root Directory to top level (null)
print("Updating rootDirectory to top level...")
update_res = requests.patch(
    f"https://api.vercel.com/v9/projects/{PROJECT_ID}",
    headers=headers,
    json={"rootDirectory": None}
)
print(f"Update status: {update_res.status_code}")

# 3. Trigger Redeploy
print("Triggering redeploy...")
# To redeploy the latest commit, we need to find the latest deployment and 'alias' it or create a new one.
# Simplest way via API is to POST to deployments with the project ID and commit info if possible, 
# but usually, just updating the settings and waiting for a hook works. 
# Let's try to trigger a manual deployment of the 'main' branch.

deploy_res = requests.post(
    "https://api.vercel.com/v13/deployments",
    headers=headers,
    json={
        "name": PROJECT_NAME,
        "gitSource": {
            "type": "github",
            "repoId": "888144141", # Need to find the repo ID or use project ID
            "ref": "main"
        }
    }
)

if deploy_res.status_code == 200:
    print(f"Deployment triggered: {deploy_res.json().get('url')}")
else:
    print(f"Deployment trigger failed: {deploy_res.text}")
    # Alternative: Trigger via 'redeploy' of last deployment
    print("Attempting to redeploy last known deployment...")
    last_deploy_res = requests.get(f"https://api.vercel.com/v6/deployments?projectId={PROJECT_ID}&limit=1", headers=headers)
    deployments = last_deploy_res.json().get('deployments', [])
    if deployments:
        last_id = deployments[0]['uid']
        redeploy = requests.post(f"https://api.vercel.com/v13/deployments?deploymentId={last_id}", headers=headers)
        print(f"Redeploy status: {redeploy.status_code}")


Searching for project: nexus-prime


Found Project: nexus-prime (prj_Jx94qNzMMYLymbM3ogYNxcmZIuAG)
Updating rootDirectory to top level...


Update status: 200
Triggering redeploy...


Deployment trigger failed: {"error":{"code":"incorrect_git_source_info","message":"The provided GitHub repository does not contain the requested branch or commit reference. Please ensure the repository is not empty."}}
Attempting to redeploy last known deployment...


Redeploy status: 400
